In [1]:
# =============================================================================
# Cell 1 - bootstrap. Loads the prepared frame and the source/target split.
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess, hashlib, time
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, conformal
import numpy as np, pandas as pd

iot=pd.read_parquet(config.PROC_DIR/'ciciot2023_prepared.parquet')
sp =pd.read_parquet(config.PROC_DIR/'ciciot2023_split.parquet')
iot['side']=sp['side'].values; iot['partition']=sp['partition'].values
FEATS=json.loads((config.REPORTS_DIR/'ciciot2023_prepared_fingerprint.json').read_text())['features']
rec=json.loads((config.REPORTS_DIR/'focal_class_record_ciciot2023.json').read_text())
FOCAL=rec['focal_class']

# canonical class order, fixed here and recorded so every downstream notebook agrees
CLASSES=sorted(iot['family'].unique().tolist())
c2i={c:i for i,c in enumerate(CLASSES)}; K=len(CLASSES)
iot['y']=iot['family'].map(c2i).astype(np.int64)     # INTEGER labels (MLP crashes on strings)
print('classes:', CLASSES, '| K =', K, '| focal:', FOCAL, f'(index {c2i[FOCAL]})')
print('features:', len(FEATS), '| frame:', iot.shape)
print(iot['partition'].value_counts().to_string())

PROBS_DIR=config.DATA_DIR/'ciciot_probs'; PROBS_DIR.mkdir(parents=True, exist_ok=True)
SEEDS=config.SEEDS
print('\nseeds:', SEEDS)


Mounted at /content/drive
classes: ['Benign', 'BruteForce', 'DDoS', 'DoS', 'Mirai', 'Recon', 'Spoofing', 'Web'] | K = 8 | focal: Web (index 7)
features: 44 | frame: (1510142, 52)
partition
train              634309
target_pool        453045
source_cal_pool    158547
probcal            158547
val                105694

seeds: [42, 1337, 2024, 7, 91, 512, 6021, 88, 3407, 12345]


In [2]:
# =============================================================================
# Cell 2 - model panel. Fixed architecture-appropriate hyperparameters rather
# than a per-dataset macro-F1 grid, to bound compute on a 634k-row training
# partition. This is the same deviation already logged for CIC-IDS2017 (nb12)
# and UGR'16 (nb16), and is recorded again below for this dataset.
# Probability calibration is one-vs-rest isotonic on D_probcal, source only,
# renormalised to sum to one (preregistration section 6).
# =============================================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

def make_model(arch, seed):
    if arch=='rf':
        return RandomForestClassifier(n_estimators=100, min_samples_leaf=5, n_jobs=-1,
                                      random_state=seed, class_weight=None)
    if arch=='xgb':
        # do NOT pass num_class/objective: let XGBClassifier infer them from y
        return XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.2,
                             tree_method='hist', n_jobs=-1, random_state=seed,
                             verbosity=0)
    if arch=='mlp':
        return MLPClassifier(hidden_layer_sizes=(128,64), max_iter=40, early_stopping=True,
                             n_iter_no_change=5, random_state=seed)
    raise ValueError(arch)

def fit_isotonic_ovr(P_cal, y_cal, K):
    """One-vs-rest isotonic on the probability-calibration partition."""
    cals=[]
    for k in range(K):
        ir=IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
        ir.fit(P_cal[:,k], (y_cal==k).astype(float))
        cals.append(ir)
    return cals

def apply_isotonic(cals, P):
    G=np.column_stack([cals[k].predict(P[:,k]) for k in range(len(cals))])
    G=np.clip(G, 1e-12, None)
    return (G / G.sum(axis=1, keepdims=True)).astype(np.float32)   # renormalised, section 6

tr = iot[iot.partition=='train']; va = iot[iot.partition=='val']
pc = iot[iot.partition=='probcal']; sc = iot[iot.partition=='source_cal_pool']
tg = iot[iot.partition=='target_pool']
Xtr, ytr = tr[FEATS].to_numpy(np.float32), tr['y'].to_numpy()
Xva, yva = va[FEATS].to_numpy(np.float32), va['y'].to_numpy()
Xpc, ypc = pc[FEATS].to_numpy(np.float32), pc['y'].to_numpy()
Xsc      = sc[FEATS].to_numpy(np.float32)
Xtg      = tg[FEATS].to_numpy(np.float32)
print('train', Xtr.shape, '| val', Xva.shape, '| probcal', Xpc.shape,
      '| src_cal_pool', Xsc.shape, '| target_pool', Xtg.shape)
print('\nclass counts in train:'); print(pd.Series(ytr).value_counts().sort_index()
      .rename(index={i:c for i,c in enumerate(CLASSES)}).to_string())
print('\nEstimated runtime: 30 fits on 634k x 44. Expect roughly 45 to 90 minutes.')


train (634309, 44) | val (105694, 44) | probcal (158547, 44) | src_cal_pool (158547, 44) | target_pool (453045, 44)

class counts in train:
Benign         25200
BruteForce      5489
DDoS          273943
DoS           100868
Mirai          75824
Recon          92237
Spoofing       50311
Web            10437

Estimated runtime: 30 fits on 634k x 44. Expect roughly 45 to 90 minutes.


In [3]:
# =============================================================================
# Cell 3 - fit, calibrate, persist. Saves calibrated probabilities for the source
# calibration pool and the whole target pool, in canonical class order, as float32.
# Resumable: a model whose npz already exists is skipped.
# =============================================================================
perf=[]
t0=time.time()
for arch in ['rf','xgb','mlp']:
    for seed in SEEDS:
        out = PROBS_DIR/f'ciciot2023__{arch}__seed{seed}.npz'
        if out.exists():
            print(f'  skip (exists) {out.name}'); continue
        t1=time.time()
        # scaler is bound per model, never left over from a previous loop iteration;
        # default-arg binding also freezes m and scaler at definition time.
        scaler = StandardScaler().fit(Xtr) if arch=='mlp' else None
        Xt = scaler.transform(Xtr).astype(np.float32) if scaler is not None else Xtr
        m=make_model(arch, seed).fit(Xt, ytr)
        def P(X, _m=m, _s=scaler):
            Z = _s.transform(X).astype(np.float32) if _s is not None else X
            return _m.predict_proba(Z)
        Pva, Ppc = P(Xva), P(Xpc)
        f1_raw = f1_score(yva, Pva.argmax(1), average='macro')
        cals = fit_isotonic_ovr(Ppc, ypc, K)
        Pva_c = apply_isotonic(cals, Pva)
        f1_cal = f1_score(yva, Pva_c.argmax(1), average='macro')
        np.savez_compressed(out,
            srcpool=apply_isotonic(cals, P(Xsc)),
            target =apply_isotonic(cals, P(Xtg)),
            classes=np.array(CLASSES))
        perf.append({'arch':arch,'seed':seed,'val_macro_f1_raw':round(float(f1_raw),4),
                     'val_macro_f1_cal':round(float(f1_cal),4),
                     'secs':round(time.time()-t1,1)})
        print(f'  {arch} seed {seed}: raw F1 {f1_raw:.4f} | cal F1 {f1_cal:.4f} '
              f'| {time.time()-t1:.0f}s | {out.stat().st_size/1e6:.1f} MB')
print(f'\ntotal {time.time()-t0:.0f}s')
mp=pd.DataFrame(perf)
if len(mp):
    print('\nmean macro-F1 by architecture (calibrated):')
    print(mp.groupby('arch')[['val_macro_f1_raw','val_macro_f1_cal']].mean().round(4).to_string())
    mp.to_csv(config.REPORTS_DIR/'model_performance_ciciot2023.csv', index=False)
else:
    print('all models already present; performance table unchanged')
print('\nprob files:', len(sorted(PROBS_DIR.glob('*.npz'))))


  rf seed 42: raw F1 0.8664 | cal F1 0.8783 | 157s | 3.2 MB
  rf seed 1337: raw F1 0.8629 | cal F1 0.8760 | 153s | 3.3 MB
  rf seed 2024: raw F1 0.8672 | cal F1 0.8785 | 157s | 3.2 MB
  rf seed 7: raw F1 0.8668 | cal F1 0.8775 | 160s | 3.4 MB
  rf seed 91: raw F1 0.8694 | cal F1 0.8809 | 148s | 3.3 MB
  rf seed 512: raw F1 0.8661 | cal F1 0.8779 | 149s | 3.3 MB
  rf seed 6021: raw F1 0.8643 | cal F1 0.8781 | 148s | 3.3 MB
  rf seed 88: raw F1 0.8616 | cal F1 0.8784 | 152s | 3.3 MB
  rf seed 3407: raw F1 0.8671 | cal F1 0.8793 | 153s | 3.2 MB
  rf seed 12345: raw F1 0.8691 | cal F1 0.8788 | 153s | 3.3 MB
  xgb seed 42: raw F1 0.8930 | cal F1 0.8926 | 234s | 2.1 MB
  xgb seed 1337: raw F1 0.8930 | cal F1 0.8926 | 241s | 2.1 MB
  xgb seed 2024: raw F1 0.8930 | cal F1 0.8926 | 235s | 2.1 MB
  xgb seed 7: raw F1 0.8930 | cal F1 0.8926 | 237s | 2.1 MB
  xgb seed 91: raw F1 0.8930 | cal F1 0.8926 | 236s | 2.1 MB
  xgb seed 512: raw F1 0.8930 | cal F1 0.8926 | 234s | 2.1 MB
  xgb seed 6021: ra

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


  mlp seed 42: raw F1 0.7657 | cal F1 0.7650 | 173s | 11.7 MB


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


  mlp seed 1337: raw F1 0.7689 | cal F1 0.7668 | 183s | 8.8 MB


...

  mlp seed 512: raw F1 0.7653 | cal F1 0.7646 | 197s | 5.7 MB
  mlp seed 6021: raw F1 0.7614 | cal F1 0.7540 | 190s | 8.7 MB


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


  mlp seed 88: raw F1 0.7473 | cal F1 0.7471 | 172s | 5.5 MB
  mlp seed 3407: raw F1 0.6825 | cal F1 0.6831 | 49s | 6.3 MB


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


  mlp seed 12345: raw F1 0.7633 | cal F1 0.7616 | 170s | 12.2 MB

total 5340s

mean macro-F1 by architecture (calibrated):
      val_macro_f1_raw  val_macro_f1_cal
arch                                    
mlp             0.7364            0.7362
rf              0.8661            0.8784
xgb             0.8930            0.8926

prob files: 30


In [ ]:
# =============================================================================
# Cell 4 - record and commit. Probability arrays are gitignored (they live in
# data/); only the performance table and the class-order record are committed.
# =============================================================================
(config.REPORTS_DIR/'ciciot2023_model_record.json').write_text(json.dumps({
  'dataset':'ciciot2023','classes_canonical_order':CLASSES,'focal_class':FOCAL,
  'focal_index':c2i[FOCAL],'n_features':len(FEATS),'seeds':list(SEEDS),
  'architectures':['rf','xgb','mlp'],
  'calibration':'one-vs-rest isotonic on D_probcal (source only), renormalised (section 6)',
  'hyperparameters':'fixed architecture-appropriate settings, not a per-dataset macro-F1 grid, '
                    'to bound compute on a 634k-row training partition; same deviation as nb12 '
                    '(CIC-IDS2017) and nb16 (UGR16), logged in deviations.md',
  'probs_dir':str(PROBS_DIR),
  'probs_contents':'srcpool = D_probcal-calibrated probabilities on the source calibration pool; '
                   'target = same on the full target pool; both in canonical class order, float32'
}, indent=2, default=str))

dev=config.REPORTS_DIR/'deviations.md'
entry = ('\n## nb33 (CIC-IoT-2023) - hyperparameters\n'
         'Fixed architecture-appropriate hyperparameters (RF/XGB/MLP) chosen for dataset scale '
         'rather than a per-dataset macro-F1 selection grid (section 5), to bound compute on a '
         '634,309-row training partition. Same deviation already logged for CIC-IDS2017 (nb12) '
         'and UGR16 (nb16). Recorded before any coverage.\n')
if 'nb33 (CIC-IoT-2023)' not in dev.read_text():
    dev.write_text(dev.read_text().rstrip()+'\n'+entry); print('logged deviation')
else: print('deviation already logged')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb33: CIC-IoT-2023 model panel and isotonic calibration, probabilities cached (no coverage)')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


logged deviation
